# TSP and Integer Programming
Based on Lab 10 of ENGRI 1101 Fall 2019





**Objectives:**
- Introduce the students to an integer programming formulation of the traveling salesman problem
- Demonstrate the use of the simplex method to solve two linear relaxations of the traveling salesman problem
- Give students an appreciation of the difficulty of solving optimization problems exactly
- Practice using branch and bound to solve integer programs

**Reading:** Chapter 16, Integer Programming and the TSP

**Names of group members:**

**Who will be "driving" for the first portion of this lab: (write their name and NetID here)**

_As a reminder, only the person driving should touch the keyboard. Everyone in the group must read the lab and help answer the questions. The driver must still discuss questions and answers with the group; they cannot write down answers without discussion, even if they already know them._

In [ ]:
# imports -- don't forget to run this cell
from ortools.linear_solver import pywraplp as OR
import itertools
import pandas as pd

## Part 1: Lower Bounds and the Minimum Spanning Tree

If one is to solve a minimization Integer Program, having lower bounds can help considerably. One way to get a lower bound is to use an LP relaxation of the problem. For the TSP, we first construct a correct integer programming formulation of the problem (see, e.g., the prelab formulation), and then obtain the LP relaxation by dropping the integrality constraints.

**Q1.1:** Why does the optimal solution to the LP relaxation provide a lower bound on the optimal solution to the TSP?

**A:**

_Write your answer here._

Using an LP relaxation is not the only possible way of getting a lower bound for a problem. For example, another lower bound for the TSP is the cost of the minimum spanning tree on the same graph. Complete the following argument that this is the case: If you start with a tour, and delete one edge, the remaining graph is connected, i.e., there is a path between any two nodes in the graph.

**Q1.2:** Why does this imply that the cost of a minimum spanning tree is a lower bound on the cost of an optimal tour?

**A:**

_Write your answer here._

In the prelab, you were asked to show that the minimum cost of a fractional 2-matching is a lower bound on the optimal tour length. Now consider an instance of the TSP on 6 nodes, where the distances between nodes $i$ and $j$ are given by the ($i,j$)-th entry in the following matrix.

In [ ]:
tsp_6_node = pd.read_csv('data/tsp_6_node.csv', index_col=0).astype(int)
tsp_6_node.columns = tsp_6_node.columns.astype(int)
display(tsp_6_node)

We can also depict this instance graphically as follows:

![title](images-lab/graph1.png)

The distance or cost of traveling between node i and j corresponds to the shortest path from $i$ to $j$ in the graph above. For example $c_{16} = 6$ and the shortest route between nodes 1 and 6 is of length 6: 1 → 4 → 6.

First, let’s consider the lower bound that we get from calculating an optimal solution to the Minimum Spanning Tree Problem for this graph. Note that we do not have to consider the edges that are not drawn in the graph. 

**Q1.3:** Draw an optimal solution for the Minimum Spanning Tree Problem for the instance, and give the length of the solution.

**A:**

_Write your answer here._

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

## Part 2: The Fractional 2-Matching Lower Bound

We will now generate a second lower bound for this instance by solving the fractional 2-matching LP that corresponds to this instance. To solve the corresponding 2-matching LP, we will use OR-Tools again. Note that there are no integrality constraints, so this is indeed an LP, i.e., the fractional 2-matching LP. Run the model cell.

In [ ]:
def two_matching(graph, cuts, integer=False):
    """A model to solve tsp problems with two-matching."""
    NODES = list(graph)                      # nodes
    cost = {}                                # costs
    for i in NODES:
        for j in NODES:
            if i < j:
                cost[(i,j)] = graph.at[i,j]
    EDGES = list(cost)                       # edges
    
    # define model
    m = OR.Solver('two_matching', OR.Solver.CBC_MIXED_INTEGER_PROGRAMMING)
    
    # decision variables
    if integer:
        x = {(i,j) : m.IntVar(0, 1, ('(%s,%s)' % (i,j))) for (i,j) in EDGES}
    else:
        x = {(i,j) : m.NumVar(0, 1, ('(%s,%s)' % (i,j))) for (i,j) in EDGES}
    
    # objective function
    m.Minimize(sum(cost[i,j]*x[i,j] for (i,j) in EDGES))
    
    # subject to: degree of every node is 2
    for k in NODES:
        m.Add(sum(x[i,j] for (i,j) in EDGES if i == k) + 
              sum(x[i,j] for (i,j) in EDGES if j == k) == 2)
    
    # subject to: provided cuts
    for S in cuts:
        m.Add(sum(x[i,j] for (i,j) in EDGES if (i in S and j not in S) or 
                                               (j in S and i not in S)) >= 2)
        
    return m,x

In [ ]:
def solve(m):
    m.Solve()
    print('Solution:')
    print('Objective value =', m.Objective().Value())
    for var in m.variables():
        print(var.name(), ':',  var.solution_value())

We will start with providing an empty list of cuts.

In [ ]:
m,x = two_matching(tsp_6_node, [])
solve(m)

**Q2.1:** What does the solution look like? Is this a feasible solution? Why or why not?

**A:**

_Write your answer here._

**Q2.2:** Is it possible that if we were to solve the minimum-cost 2-matching problem (where we add the integrality constraints to the above LP) for this data set we would get a better lower bound? Why or why not?

**A:**

_Write your answer here._

**Q2.3:** Did we get a feasible tour? 

**A:**

_Write your answer here._

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

## Part 3: The Subtour Elimination LP

A third lower bound for the TSP can be obtained from the LP relaxation of an IP formulation of the TSP. In particular, we will be interested in the following linear programming problem, which is a relaxation of an IP for the TSP:

$$(1)\quad \text{minimize} \sum_{i=1}^{n-1}\sum_{j = i+1}^n c_{ij}x_{ij}$$
$$\text{subject to}$$
$$(2)\quad \sum_{j=1}^{i-1} x_{ji} + \sum_{j = i+1}^n x_{ij} = 2, \forall i = 1,\dots,n$$
$$(3)\quad \sum_{\{i,j\} \in \delta(S)}x_{ij} \geq 2, \forall S \subset V$$
$$(4)\quad 0 \leq x_{ij} \leq 1, \forall i = 1,\dots,n-1, j = i+1,\dots,n$$

This linear program is called the Subtour Elimination LP (recall that with the integrality constraints the Subtour Elimination LP is a correct integer program for the TSP problem).

**Q3.1:** Is the optimal solution to the fractional 2-matching problem that you just found a feasible solution to this linear program? Why or why not?

**A:**

_Write your answer here._

One problem with the subtour elimination constraints is that we add a constraint of the form (3) for each set of the vertices S (disregarding the set of no vertices S = {} and the set S of all vertices S = V , since the subtour elimination constraints don’t make sense). 

**Q3.2:** If we have n = 10 vertices, how many subtour elimination constraints would we need to write? What if n = 100?

**A:**

_Write your answer here._

For $n = 100$ we’d need something like $2^{100}$ constraints, which is known technically as “way too many.” (You can do a little bit better by being clever, but not substantially). In practice, we need a method to add subtour elimination constraints one at a time, without writing them all down. In fact, one can use a maximum flow algorithm to automatically check if the constraints (3) are all satisfied, and if not, find a violated constraint, without having to check them all, one at a time (we are not going to explain how this can be done). This leads to the following algorithm to solve this very large LP. Solve the minimum-cost fractional 2-matching problem (using the simplex method). Check if this solution satisfies all of the constraints (3). If yes, then this is an optimal solution to the Subtour Elimination LP.

**Q3.3:** Why?

**A:**

_Write your answer here._

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

If not, then the algorithm will find a constraint that is violated. This constraint is added to the fractional 2-matching LP, and the algorithm solves this new LP. Now for this new solution, check again if all of the constraints (3) are satisfied. If yes, then you have the optimal solution to the Subtour Elimination LP. Otherwise continue this procedure of adding violated constraints until an optimal solution is found.

Previously, we found that the subtour elimination constraints were not satisfied, so our current solution (the one you most recently drew, by solving the fractional 2-matching LP without any subtour constraints) is not feasible for the subtour LP. Hence, we have to find and add a violated subtour constraint corresponding to a set of vertices S. You already found this at the end of page 4!

**Q3.4:** Write down this set S.

**A:**

_Write your answer here._

You will now add this constraint to your LP. This can be done by updating ```cuts```. This variable contains a list of cuts. Right now, the list of cuts is empty. For example, if $S = \{1,2,5\}$, you would update ```cuts``` with `cuts = [[1,2,5]]` 

In [ ]:
# TODO: Update cuts with the cut S you found in Q3.4



Solve the updated LP. 

In [ ]:
m,x = two_matching(tsp_6_node, cuts)
solve(m)

**Q3.5:** Give the objective value of the new optimal solution and draw the graph with this solution.

**A:**

_Write your answer here._

If you can find any additional violated subtour constraints, repeat the above. When you are done, you have found an optimal solution to the Subtour Elimination LP linear program, even though you didn’t have to write out all sub-tour elimination constraints explicitly!


**Q3.6:** Draw your solution to the subtour LP and give the objective value.
  
**A:**

_Write your answer here._

### Key Takeaways

Nice work! In this lab, we studied the traveling salesman problem and in particular, looked at *lower bounds*. Because the TSP is hard to solve exactly, a good lower bound is valuable: it tells us how close a candidate tour is to optimal, which ensures branch and bound can run more efficiently.

We saw several lower bounds: the minimum spanning tree and the fractional 2-matching LP, but had the issue of subtours. To rule those out, we added subtour elimination constraints, which forced at least 2 units of flow across the boundary of every vertex set $S$. Together with the integrality constraints, this is a valid IP formulation of the TSP. The catch is that there are exponentially many such constraints&mdash;about $2^n$&mdash;so we cannot write them all down. Instead, we used a *cutting-plane* approach: solve the relaxation, check whether any subtour constraint is violated, add only the violated ones, and repeat. This let us solve the Subtour Elimination LP to optimality without listing every single constraint.